
# 05 - Data Validation & Quality Checks
# Validate Bronze, Silver and Gold Data Layers

 Validar Bronze

In [0]:
%sql

SELECT
    execution_date,
    source_api,
    status_code,
    extracted_at,
    pipeline_run_id
FROM mlb_project.bronze_mlb_schedule_raw
ORDER BY extracted_at DESC
LIMIT 20;

 Qué valida:
Confirma que la API se extrajo y se guardó en Bronze.

Validar Silver

In [0]:
%sql
SELECT
    game_pk,
    game_date,
    away_team_name,
    home_team_name,
    away_score,
    home_score,
    game_status,
    venue_name,
    updated_at
FROM mlb_project.silver_mlb_games
ORDER BY game_date DESC, home_team_name
LIMIT 30;

Qué valida:
Confirma que el JSON fue transformado en una tabla limpia.

Validar Gold

In [0]:
%sql
SELECT *
FROM mlb_project.gold_mlb_daily_summary
ORDER BY game_date DESC
LIMIT 20;

Qué valida:
Confirma que se creó el resumen analítico.

Validar logs

In [0]:
%sql
SELECT
    pipeline_name,
    pipeline_run_id,
    layer,
    status,
    message,
    execution_date,
    records_processed,
    started_at,
    ended_at,
    duration_seconds
FROM mlb_project.pipeline_logs
ORDER BY started_at DESC
LIMIT 30;

Qué valida:
Confirma si cada etapa corrió correctamente o si hubo errores.

Conteo por capa

In [0]:
%sql
SELECT 'Bronze' AS layer, COUNT(*) AS total_records
FROM mlb_project.bronze_mlb_schedule_raw

UNION ALL

SELECT 'Silver' AS layer, COUNT(*) AS total_records
FROM mlb_project.silver_mlb_games

UNION ALL

SELECT 'Gold' AS layer, COUNT(*) AS total_records
FROM mlb_project.gold_mlb_daily_summary;

Qué valida:
Muestra cuántos registros tiene cada capa.

Calidad de datos en Silver

In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN game_pk IS NULL THEN 1 ELSE 0 END) AS null_game_pk,
    SUM(CASE WHEN game_date IS NULL THEN 1 ELSE 0 END) AS null_game_date,
    SUM(CASE WHEN home_team_name IS NULL THEN 1 ELSE 0 END) AS null_home_team,
    SUM(CASE WHEN away_team_name IS NULL THEN 1 ELSE 0 END) AS null_away_team
FROM mlb_project.silver_mlb_games;

Qué valida:
Revisa campos importantes con valores nulos.

Duplicados en Silver

In [0]:
%sql
SELECT
    game_pk,
    COUNT(*) AS total_records
FROM mlb_project.silver_mlb_games
GROUP BY game_pk
HAVING COUNT(*) > 1;

Juegos por estado

In [0]:
%sql
SELECT
    game_status,
    COUNT(*) AS total_games
FROM mlb_project.silver_mlb_games
GROUP BY game_status
ORDER BY total_games DESC;